1. Data Cleaning Business Rules
BR-01: Duplicate Movie Handling
Rule: A movie is considered duplicate if title + release_year is repeated.
Action:
Keep the record with highest rating
If ratings are equal, keep the one with highest gross

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('../data/raw/imdb_movies.csv')

In [5]:
df = df.sort_values(
    ['rating','gross_million'],
    ascending=False
)

df = df.drop_duplicates(
    subset=['title','release_year'],
    keep='first'
)
print("Done")

Done


BR-02: Missing Movie ID
Rule: movie_id must be unique and non-null.
Action:
Auto-generate a surrogate key during ETL if missing.

In [6]:
max_id = df['movie_id'].max()

missing_rows = df['movie_id'].isnull()

df.loc[
    missing_rows,
    'movie_id'
] = range(
    int(max_id)+1,
    int(max_id)+1+missing_rows.sum()
)
print("Auto-generated surrogate key")

Auto-generated surrogate key


BR-03: Missing Rating
Rule: Ratings must be between 0 and 10.
Action:
Replace NULL ratings with genre-wise average rating.

In [7]:
genre_avg = (
    df.groupby('genre')['rating']
      .transform('mean')
)

df['rating'] = df['rating'].fillna(genre_avg)
print("Rating Imputation")

Rating Imputation


BR-04: Missing Director / Actor
Rule: Director and lead actor are mandatory descriptive attributes.
Action:
Replace NULL values with 'Unknown'.

In [8]:
df['director'] = df['director'].fillna('Unknown')

df['lead_actor'] = df['lead_actor'].fillna('Unknown')
print("Replaced null values")

Replaced null values


In [9]:
df.to_csv(
    '../data/clean/imdb_movies_clean.csv',
    index=False
)

In [10]:
df.isnull().sum()

movie_id            0
title               0
genre               0
rating              0
budget_million    928
gross_million     913
director            0
lead_actor          0
release_year      610
dtype: int64

In [11]:
df.duplicated(
    subset=['title', 'release_year']
).sum()

np.int64(0)